# 🧠 Kairos CIDA v4: Latent Policy RL Training (Teacher-in-the-Loop)

This notebook implements **Latent Policy Reinforcement Learning (LPRL)**.

### 🎯 The Philosophy:
CIDA is a **blind agent** operating inside the hidden layers of the model. It does not know the answer key. Its task is to perform latent moves (shifting vectors via $\delta$) and learn **how to search and resolve contradictions** (action policies).

The **Teacher (7B Model)** acts as a frozen critic/environment. Guided by a rich instruction prompt, the Teacher evaluates the *reasoning process* generated by the student, scoring it on logic, consistency, and constraint satisfaction.

We optimize CIDA using **Policy Gradient (REINFORCE)** based on the Teacher's logical rewards. This prevents memorization and forces CIDA to learn generalized *cognitive navigation strategies* (how to walk in the gradient space).

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets accelerate bitsandbytes huggingface_hub tqdm

import getpass, sys, os
token = getpass.getpass("GitHub PAT: ").strip()
if token:
    !git clone https://{token}@github.com/Kairatzh/CIDA-plugin.git
    sys.path.append("CIDA-plugin")
    !pip install -q -e CIDA-plugin
else:
    !git clone https://github.com/Kairatzh/CIDA-plugin.git
    sys.path.append("CIDA-plugin")
    !pip install -q -e CIDA-plugin

import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

In [ ]:
import os, gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from cida_plugin.config import CIDAPluginConfig
from cida_plugin.core import CIDAPlugin
from cida_plugin.projector import LatentAEProjector

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
STUDENT_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
TEACHER_NAME = "Qwen/Qwen2.5-7B-Instruct"

print("Loading Tokenizers...")
tokenizer = AutoTokenizer.from_pretrained(STUDENT_NAME, padding_side="left")
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

# Config for teacher quantization
bnb_teacher = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")

print("Loading Student LLM (1.5B) in native bfloat16 (Fast & Accurate)...")
student_llm = AutoModelForCausalLM.from_pretrained(
    STUDENT_NAME, 
    torch_dtype=torch.bfloat16, 
    device_map="auto"
)
student_llm.eval()

print("Loading Teacher LLM (7B) in 4-bit NF4...")
teacher_llm = AutoModelForCausalLM.from_pretrained(
    TEACHER_NAME, 
    quantization_config=bnb_teacher, 
    device_map="auto"
)
teacher_llm.eval()

# Initialize CIDA config
cfg = CIDAPluginConfig(
    d_input=student_llm.config.hidden_size, d_hidden=1024, d_message=1024,
    num_agents=4, num_classes=2, max_rounds=4,
    num_attn_heads=8, use_perspective_projector=True,
    anonymous_messages=True, sparse_comm_k=3, use_dynamic_trust=True
)
cida = CIDAPlugin(cfg).to(DEVICE).float()
proj = LatentAEProjector(cfg.d_hidden, student_llm.config.hidden_size, cfg.num_agents, num_plan=8).to(DEVICE)

# Locate CIDA pre-trained SFT weights (either local or drive)
drive_path = "/content/drive/MyDrive/cida_v4_universal_weights"
local_path = "cida_v4_universal_weights"
v4_dir = drive_path if os.path.exists(os.path.join(drive_path, "pytorch_model.bin")) else (local_path if os.path.exists(os.path.join(local_path, "pytorch_model.bin")) else None)

if v4_dir:
    print(f"Loading pre-trained CIDA weights from: {v4_dir} (Warm start)")
    try:
        cida.load_state_dict(torch.load(os.path.join(v4_dir, "pytorch_model.bin"), map_location="cpu"))
        proj.load_state_dict(torch.load(os.path.join(v4_dir, "projector.pt"), map_location="cpu"))
        print("✅ Pre-trained CIDA weights loaded successfully! Ready for RL alignment.")
    except Exception as e:
        print(f"Failed to load weights: {e}. Falling back to training from scratch.")
else:
    print("ℹ️ Pre-trained weights not found. Initializing fresh CIDA from scratch.")

In [ ]:
import torch.optim as optim
import json, re

# 1. Teacher Critique Prompt
TEACHER_SYSTEM_PROMPT = (
    "You are an expert logic critic. Evaluate the Student's reasoning path.\n"
    "Analyze if the student satisfied constraints, avoided local biases, and reasoned correctly.\n"
    "Provide your evaluation in JSON format with exactly two fields:\n"
    "1. 'score': a float between 0.0 (illogical) and 1.0 (perfectly correct).\n"
    "2. 'hint': a short, 1-sentence hint/clue pointing to the correct logic (e.g. 'Focus on Friday departure constraint').\n"
    "Output ONLY the raw JSON block, nothing else."
)

def make_teacher_prompt(q, student_response, correct_answer):
    return (
        f"<|im_start|>system\n{TEACHER_SYSTEM_PROMPT}\n<|im_end|>\n"
        f"<|im_start|>user\n"
        f"Question: {q}\n"
        f"Correct Option Key: {correct_answer}\n"
        f"Student's Reasoning & Output: {student_response}\n\n"
        f"Evaluate the logical coherence. Output JSON: "
        f"<|im_end|>\n<|im_start|>assistant\n"
    )

def evaluate_with_teacher(question, student_response, correct_answer):
    t_prompt = make_teacher_prompt(question, student_response, correct_answer)
    t_enc = tokenizer(t_prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        t_ids = teacher_llm.generate(**t_enc, max_new_tokens=60, do_sample=False)
        t_ans = tokenizer.decode(t_ids[0, t_enc.input_ids.size(1):], skip_special_tokens=True).strip()

    # Parse JSON output with robust regex fallback
    try:
        # Clean markdown formatting if present
        t_ans_clean = t_ans.replace("```json", "").replace("```", "").strip()
        match = re.search(r'\{.*\}', t_ans_clean, re.DOTALL)
        data = json.loads(match.group(0) if match else t_ans_clean)
        score = min(max(float(data.get("score", 0.0)), 0.0), 1.0)
        hint = data.get("hint", "Focus on the logical constraints of the question.").strip()
        return score, hint
    except Exception as parse_err:
        # Regex fallback extraction for 'score'
        score_match = re.search(r'"score":\s*([0-9.]+)', t_ans)
        score = float(score_match.group(1)) if score_match else (0.5 if correct_answer in student_response else 0.0)
        hint = "Focus on the logical constraints of the question."
        return score, hint

# 2. RL Optimizer & Cosine Annealing Scheduler
optimizer = optim.Adam(list(cida.parameters()) + list(proj.parameters()), lr=2e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5000, eta_min=1e-6)

def train_rl_step(sample):
    q = sample["question"]
    correct_ans = sample["answer"]

    prompt = f"<|im_start|>system\nSolve logically.\n<|im_end|>\n<|im_start|>user\n{q}\n<|im_end|>\n<|im_start|>assistant\n"
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    # Student Forward Pass
    with torch.no_grad():
        out = student_llm(input_ids=enc.input_ids, attention_mask=enc.attention_mask, output_hidden_states=True)
        compute_dtype = out.hidden_states[-1].dtype
        h_base = out.hidden_states[-1][0, -1].float().unsqueeze(0)
        p_emb = student_llm.get_input_embeddings()(enc.input_ids).detach()

    # CIDA agent action selection (continuous latent shift)
    co = cida(pooled=h_base)
    plan = proj(co["state_final"].s) # shape: (1, 8, d_llm)

    disagreement = co["disagreement"].mean()

    # Generate answer from student model
    with torch.no_grad():
        cat = torch.cat([p_emb.to(compute_dtype), plan.to(compute_dtype)], dim=1).to(compute_dtype)
        pm = torch.ones(1, 8, dtype=enc.attention_mask.dtype, device=DEVICE)
        gm = torch.cat([enc.attention_mask, pm], dim=1)
        ids = student_llm.generate(inputs_embeds=cat, attention_mask=gm, max_new_tokens=100, do_sample=True, temperature=0.7)
        student_ans = tokenizer.decode(ids[0], skip_special_tokens=True).strip()

    # Get score and verbal hint from SOTA Teacher Model
    reward, hint_text = evaluate_with_teacher(q, student_ans, correct_ans)

    # Get embedding of the teacher's hint under the student's embedding layer
    hint_tokens = tokenizer(hint_text, return_tensors="pt").input_ids.to(DEVICE)
    with torch.no_grad():
        hint_emb = student_llm.get_input_embeddings()(hint_tokens).mean(dim=1).float() # shape: (1, d_llm)

    # Compute Critique Alignment Loss: maximize cosine similarity between plan tokens and hint embedding
    plan_mean = plan.mean(dim=1).float() # shape: (1, d_llm)
    cosine_sim = torch.nn.functional.cosine_similarity(plan_mean, hint_emb)
    align_loss = 1.0 - cosine_sim.mean()

    # Compute L2 Regularization to keep plan vector within natural embedding manifold
    reg_loss = torch.mean(plan ** 2)

    # Optimize: Combined Policy Gradient, Critique Alignment, and L2 Embedding Regularization
    optimizer.zero_grad()
    rl_loss = disagreement * (1.0 - reward) + 0.05 * align_loss + 0.01 * reg_loss
    rl_loss.backward()
    optimizer.step()

    # Step the learning rate scheduler
    scheduler.step()

    return reward, hint_text, rl_loss.item(), student_ans

print("✅ RL Policy Training Engine with Critique Alignment compiled!")

In [ ]:
from datasets import load_dataset
import random
import gc

# 1. Dataset loading (Scales up to 5000 samples for the full RL run)
MAX_SAMPLES = 5000

print(f"Loading up to {MAX_SAMPLES} OpenThoughts-114k SOTA tasks...")
try:
    ot_dataset = load_dataset("open-thoughts/OpenThoughts-114k", "metadata", split="train", streaming=True)
    rl_dataset = []
    for i, item in enumerate(ot_dataset):
        if len(rl_dataset) >= MAX_SAMPLES: break
        q = item.get("problem", "")
        cot = item.get("deepseek_reasoning", "")
        ans = item.get("deepseek_solution", "")
        domain = item.get("domain", "")

        # Keep reasoning/logic tasks with verifiable answers from math, logic, puzzles or science domains
        if domain in ["math", "puzzles", "logic", "science"] and len(q) > 30 and len(ans) > 0 and len(cot) > 100:
            rl_dataset.append({
                "question": q,
                "answer": ans
            })
    print(f"✅ Loaded {len(rl_dataset)} SOTA tasks from OpenThoughts-114k!")
except Exception as e:
    print(f"Error loading OpenThoughts-114k: {e}. Falling back to toy dataset.")
    rl_dataset = [
        {
            "question": "AI travel agent booking from Astana to Dubai. Strict constraints: 1. Price < $1000. 2. Depart on Friday. Flights: A: Wednesday $800, B: Friday $950, C: Friday $1200.",
            "answer": "B"
        }
    ]

print("Starting Teacher-Guided RL Policy training on OpenThoughts...")
# Put CIDA models explicitly in training mode
cida.train()
proj.train()

total_reward = 0.0
for idx, sample in enumerate(rl_dataset):
    reward, hint_text, loss, ans = train_rl_step(sample)
    total_reward += reward

    # Log training status every 50 steps
    if (idx + 1) % 50 == 0:
        current_lr = scheduler.get_last_lr()[0]
        print(f"Step {idx+1}/{len(rl_dataset)} | Avg Reward: {total_reward/(idx+1):.2f} | Loss: {loss:.4f} | LR: {current_lr:.2e}")
        print(f"  -> Teacher Hint: '{hint_text}'")
        print(f"  -> Sample Student Answer: {ans[:120]}...")
        print("-" * 60)

    # Periodic memory reclamation to prevent OOM fragmentation on long 5000-sample runs
    if (idx + 1) % 100 == 0:
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
import copy

print("Loading pre-trained CIDA v4 (SFT) for comparison...")
# Initialize a duplicate of the architecture
cida_v4 = copy.deepcopy(cida)
proj_v4 = copy.deepcopy(proj)

# Locate CIDA v4 SFT weights (either local or drive)
drive_path = "/content/drive/MyDrive/cida_v4_universal_weights"
local_path = "cida_v4_universal_weights"
v4_dir = drive_path if os.path.exists(os.path.join(drive_path, "pytorch_model.bin")) else (local_path if os.path.exists(os.path.join(local_path, "pytorch_model.bin")) else None)

if v4_dir:
    try:
        cida_v4.load_state_dict(torch.load(os.path.join(v4_dir, "pytorch_model.bin"), map_location="cpu"))
        proj_v4.load_state_dict(torch.load(os.path.join(v4_dir, "projector.pt"), map_location="cpu"))
        print("✅ CIDA v4 (SFT) weights loaded successfully.")
    except Exception as e:
        print(f"Failed to load CIDA v4: {e}. Will compare Base vs CIDA v5 only.")
        cida_v4 = None
else:
    print("⚠️ CIDA v4 SFT weights not found. Will compare Base vs CIDA v5 RL only.")
    cida_v4 = None

# Grab 5 fresh logic questions from OpenThoughts (out-of-distribution, from step 150 onwards)
eval_samples = rl_dataset[150:155] if len(rl_dataset) > 155 else rl_dataset[:5]

print("\n" + "="*80)
print("      🏆 SIDE-BY-SIDE EVALUATION: BASE vs CIDA V4 (SFT) vs CIDA V5 (RL) 🏆")
print("="*80)

for idx, sample in enumerate(eval_samples):
    q = sample["question"]
    correct_ans = sample["answer"]

    prompt = f"<|im_start|>system\nSolve logically.\n<|im_end|>\n<|im_start|>user\n{q}\n<|im_end|>\n<|im_start|>assistant\n"
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    # 1. Base LLM generation (No CIDA)
    with torch.no_grad():
        ids = student_llm.generate(**enc, max_new_tokens=100, do_sample=True, temperature=0.7)
        ans_base = tokenizer.decode(ids[0, enc.input_ids.size(1):], skip_special_tokens=True).strip()
        score_base, _ = evaluate_with_teacher(q, ans_base, correct_ans)

    # 2. CIDA v4 (SFT) generation
    score_v4 = 0.0
    ans_v4 = "N/A"
    if cida_v4:
        with torch.no_grad():
            out = student_llm(input_ids=enc.input_ids, attention_mask=enc.attention_mask, output_hidden_states=True)
            compute_dtype = out.hidden_states[-1].dtype
            h_base = out.hidden_states[-1][0, -1].float().unsqueeze(0)
            co = cida_v4(pooled=h_base)
            plan = proj_v4(co["state_final"].s)

            p_emb = student_llm.get_input_embeddings()(enc.input_ids).detach()
            cat = torch.cat([p_emb.to(compute_dtype), plan.to(compute_dtype)], dim=1).to(compute_dtype)
            pm = torch.ones(1, 8, dtype=enc.attention_mask.dtype, device=DEVICE)
            gm = torch.cat([enc.attention_mask, pm], dim=1)

            ids = student_llm.generate(inputs_embeds=cat, attention_mask=gm, max_new_tokens=100, do_sample=True, temperature=0.7)
            ans_v4 = tokenizer.decode(ids[0], skip_special_tokens=True).strip()
            score_v4, _ = evaluate_with_teacher(q, ans_v4, correct_ans)

    # 3. CIDA v5 (RL-aligned) generation
    with torch.no_grad():
        out = student_llm(input_ids=enc.input_ids, attention_mask=enc.attention_mask, output_hidden_states=True)
        compute_dtype = out.hidden_states[-1].dtype
        h_base = out.hidden_states[-1][0, -1].float().unsqueeze(0)
        co = cida(pooled=h_base)
        plan = proj(co["state_final"].s)

        p_emb = student_llm.get_input_embeddings()(enc.input_ids).detach()
        cat = torch.cat([p_emb.to(compute_dtype), plan.to(compute_dtype)], dim=1).to(compute_dtype)
        pm = torch.ones(1, 8, dtype=enc.attention_mask.dtype, device=DEVICE)
        gm = torch.cat([enc.attention_mask, pm], dim=1)

        ids = student_llm.generate(inputs_embeds=cat, attention_mask=gm, max_new_tokens=100, do_sample=True, temperature=0.7)
        ans_v5 = tokenizer.decode(ids[0], skip_special_tokens=True).strip()
        score_v5, _ = evaluate_with_teacher(q, ans_v5, correct_ans)

    print(f"\n📝 QUESTION {idx+1}: {q[:120]}...")
    print(f"🎯 CORRECT ANSWER: {correct_ans}")
    print("-" * 50)
    print(f"🔹 [Base LLM] Score: {score_base:.2f} | Answer: {ans_base[:150]}...")
    if cida_v4:
        print(f"🔸 [CIDA v4 SFT] Score: {score_v4:.2f} | Answer: {ans_v4[:150]}...")
    print(f"🟢 [CIDA v5 RL] Score: {score_v5:.2f} | Answer: {ans_v5[:150]}...")
    print("="*80)

In [ ]:
import random
import re
import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset

print("Loading OOD Benchmarks (BBH & FOLIO)...")
datasets_eval = {}
EVAL_SAMPLES = 10  # Fast evaluation size to run in 3-4 minutes

# 1. BBH Fallacies
try:
    ds_fallacy = load_dataset("lukaemon/bbh", "formal_fallacies", split="test")
    samples = list(ds_fallacy)
    random.seed(42)
    random.shuffle(samples)
    datasets_eval["BBH (Fallacies)"] = [{"q": f"Deductive Argument:\n{s['input']}\nIs this argument valid or invalid? Answer with just the word.", "a": s['target'].strip()} for s in samples[:EVAL_SAMPLES]]
    print(f"Loaded {len(datasets_eval['BBH (Fallacies)'])} BBH Fallacies samples.")
except Exception as e:
    print(f"Error loading BBH Fallacies: {e}")

# 2. BBH Logical Deduction
try:
    ds_bbh = load_dataset("lukaemon/bbh", "logical_deduction_five_objects", split="test")
    samples = list(ds_bbh)
    random.seed(42)
    random.shuffle(samples)
    datasets_eval["BBH (Logical)"] = [{"q": f"Logic Puzzle:\n{s['input']}\nJust answer with the correct letter.", "a": s['target'].strip("()")} for s in samples[:EVAL_SAMPLES]]
    print(f"Loaded {len(datasets_eval['BBH (Logical)'])} BBH Logical samples.")
except Exception as e:
    print(f"Error loading BBH: {e}")

# 3. FOLIO
try:
    ds_folio = load_dataset("tasksource/folio", split="validation")
    samples = list(ds_folio)
    random.seed(42)
    random.shuffle(samples)

    folio_formatted = []
    for s in samples[:EVAL_SAMPLES]:
        prems = s['premises']
        prems_str = "\n".join(prems) if isinstance(prems, list) else str(prems)
        folio_formatted.append({
            "q": f"Premises:\n{prems_str}\nConclusion: {s['conclusion']}\nIs the conclusion True, False, or Unknown? Answer with just the word.",
            "a": s['label'].strip()
        })
    datasets_eval["FOLIO (FOL)"] = folio_formatted
    print(f"Loaded {len(datasets_eval['FOLIO (FOL)'])} FOLIO samples.")
except Exception as e:
    print(f"Error loading FOLIO: {e}")

# Teacher Grading Helper Function
def evaluate_answers_with_teacher(q, correct_ans, ans_base, ans_v5):
    prompt = (
        f"<|im_start|>system\n"
        f"You are a precise grading assistant. Evaluate if the answers of Model A and Model B are correct according to the Reference Correct Answer.\n"
        f"Respond ONLY with a JSON object in this format: {{'base_correct': 0 or 1, 'cida_correct': 0 or 1}}\n"
        f"<|im_end|>\n"
        f"<|im_start|>user\n"
        f"Question:\n{q}\n\n"
        f"Reference Correct Answer:\n{correct_ans}\n\n"
        f"Model A's Answer:\n{ans_base}\n\n"
        f"Model B's Answer:\n{ans_v5}\n"
        f"<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    t_enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        t_ids = teacher_llm.generate(**t_enc, max_new_tokens=40, do_sample=False)
        t_ans = tokenizer.decode(t_ids[0, t_enc.input_ids.size(1):], skip_special_tokens=True).strip()

    base_correct = 0
    cida_correct = 0
    try:
        t_ans_clean = t_ans.replace("```json", "").replace("```", "").strip()
        match = re.search(r'\{.*\}', t_ans_clean, re.DOTALL)
        if match:
            data = json.loads(match.group(0))
        else:
            data = json.loads(t_ans_clean)
        base_correct = int(data.get("base_correct", 0))
        cida_correct = int(data.get("cida_correct", 0))
    except Exception:
        # Fallback regex
        base_match = re.search(r'"base_correct":\s*([01])', t_ans)
        cida_match = re.search(r'"cida_correct":\s*([01])', t_ans)
        if base_match: base_correct = int(base_match.group(1))
        if cida_match: cida_correct = int(cida_match.group(1))
        if not base_match and not cida_match:
            if "base_correct\": 1" in t_ans or "base_correct\":1" in t_ans: base_correct = 1
            if "cida_correct\": 1" in t_ans or "cida_correct\":1" in t_ans: cida_correct = 1
    return base_correct, cida_correct

# Evaluate active models in memory
results = {"base": {}, "cida_v5": {}}

# Temporarily set models to eval mode
student_llm.eval()
cida.eval()
proj.eval()

for ds_name, samples in datasets_eval.items():
    print(f"\nEvaluating {ds_name}...")
    correct_base = 0
    correct_cida = 0

    for idx, s in enumerate(samples):
        prompt = f"<|im_start|>system\nSolve logically.\n<|im_end|>\n<|im_start|>user\n{s['q']}\n<|im_end|>\n<|im_start|>assistant\n"
        enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)

        # Base generation
        with torch.no_grad():
            ids = student_llm.generate(**enc, max_new_tokens=60, do_sample=True, temperature=0.7)
            ans_base = tokenizer.decode(ids[0, enc.input_ids.size(1):], skip_special_tokens=True).strip()

        # CIDA generation
        with torch.no_grad():
            out = student_llm(input_ids=enc.input_ids, attention_mask=enc.attention_mask, output_hidden_states=True)
            compute_dtype = out.hidden_states[-1].dtype
            h_base = out.hidden_states[-1][0, -1].float().unsqueeze(0)
            co = cida(pooled=h_base)
            plan = proj(co["state_final"].s)

            p_emb = student_llm.get_input_embeddings()(enc.input_ids).detach()
            cat = torch.cat([p_emb.to(compute_dtype), plan.to(compute_dtype)], dim=1).to(compute_dtype)
            pm = torch.ones(1, 8, dtype=enc.attention_mask.dtype, device=DEVICE)
            gm = torch.cat([enc.attention_mask, pm], dim=1)

            ids = student_llm.generate(inputs_embeds=cat, attention_mask=gm, max_new_tokens=60, do_sample=True, temperature=0.7)
            ans_v5 = tokenizer.decode(ids[0], skip_special_tokens=True).strip()

        # Teacher Evaluation
        base_correct, cida_correct = evaluate_answers_with_teacher(s['q'], s['a'], ans_base, ans_v5)
        correct_base += base_correct
        correct_cida += cida_correct

        # Print debug for first 3 samples
        if idx < 3:
            print(f"  [Sample {idx+1}] Q: {s['q'][:100]}...")
            print(f"    Base: '{ans_base}' -> Graded: {base_correct}")
            print(f"    CIDA: '{ans_v5}' -> Graded: {cida_correct}")
            print(f"    Target: '{s['a']}'")
            print("-" * 40)

    acc_base = (correct_base / len(samples)) * 100
    acc_cida = (correct_cida / len(samples)) * 100
    results["base"][ds_name] = acc_base
    results["cida_v5"][ds_name] = acc_cida
    print(f"📊 {ds_name} -> Base: {acc_base:.1f}% | CIDA v5: {acc_cida:.1f}% (Delta: {acc_cida - acc_base:+.1f}%)")

# Plot dashboard
datasets_names = list(datasets_eval.keys())
base_accs = [results["base"][d] for d in datasets_names]
cida_accs = [results["cida_v5"][d] for d in datasets_names]
deltas = [cida_accs[i] - base_accs[i] for i in range(len(datasets_names))]

plt.figure(figsize=(12, 5))
plt.style.use('dark_background')

x = np.arange(len(datasets_names))
width = 0.35

# Subplot 1: Absolute Accuracies
ax1 = plt.subplot(1, 2, 1)
ax1.bar(x - width/2, base_accs, width, label="Base LLM (1.5B)", color="#475569")
ax1.bar(x + width/2, cida_accs, width, label="CIDA v5 (RL)", color="#10b981", edgecolor="#34d399", linewidth=1.5)
ax1.set_ylabel('Accuracy (%)')
ax1.set_title('Absolute Performance')
ax1.set_xticks(x)
ax1.set_xticklabels(datasets_names)
ax1.legend(loc='upper left', facecolor='#1e293b')
ax1.set_ylim(0, 100)
ax1.grid(axis='y', linestyle='--', alpha=0.1)

# Subplot 2: Deltas
ax2 = plt.subplot(1, 2, 2)
colors = ['#10b981' if d >= 0 else '#ef4444' for d in deltas]
ax2.bar(datasets_names, deltas, color=colors, width=0.4)
ax2.set_ylabel('Net Gain Delta (%)')
ax2.set_title('Logic Improvement Delta (CIDA v5 - Base)')
ax2.set_ylim(-20, 50)
ax2.axhline(0, color='gray', linestyle='-', linewidth=0.8)
ax2.grid(axis='y', linestyle='--', alpha=0.1)

plt.tight_layout()
plt.savefig("cida_v5_benchmark_dashboard.png")
plt.show()
print("📈 Saved dashboard to: cida_v5_benchmark_dashboard.png")